<a href="https://colab.research.google.com/github/MarkowitzMx/Programacion-para-analitica-descriptiva-y-predictiva-2026/blob/main/Copy_of_Sesion12_Evaluacion_Datos_Categoricos_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Programación para Analítica Descriptiva y Predictiva**
**Maestría en Inteligencia Artificial y Analítica de Datos**

# Sesión 12: Evaluación — Limpieza y Transformación de Datos Categóricos

**Entrega individual**

- **Nombre**: Cristóbal Lemus Rendón
- **Matrícula** 274690

Esta evaluación aplica los tres temas de la Sesión 12 (errores tipográficos y valores inconsistentes, alta cardinalidad, tipos incorrectos) a un dataset que no se trabajó en clase: **Telco Customer Churn**.


No hay una única respuesta correcta en varias de las actividades — lo que se evalúa es que la conclusión esté respaldada por el código que la sustenta, no solo la conclusión en sí.

## Preparación

In [9]:
import pandas as pd
import numpy as np

url = 'https://raw.githubusercontent.com/treselle-systems/customer_churn_analysis/master/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(url)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


---
## Actividad 1 — Formato y valores inconsistentes (15 pts)

Revisa **todas** las columnas categóricas del dataset (no elijas solo una) en busca de variantes de formato (mayúsculas, espacios) que deberían normalizarse.

In [11]:
# 1.1 — Recorremos todas las columnas categóricas (dtype object/str) del dataframe
# para inspeccionar sus valores y detectar posibles inconsistencias de formato
# (mayúsculas/minúsculas mezcladas, espacios extra, etc.)

# Seleccionamos únicamente las columnas de tipo texto (categóricas), excluyendo
# las numéricas (SeniorCitizen, tenure, MonthlyCharges) y dejando fuera TotalCharges
# porque, aunque pandas la lee como texto, en realidad es una columna numérica

columnas_categoricas = df.select_dtypes(include='object').columns.tolist()
columnas_categoricas.remove('TotalCharges')  # es numérica mal tipada, no categórica real

print("Columnas categóricas a revisar:", columnas_categoricas)
print("-" * 60)

# Recorremos cada columna categórica e imprimimos sus valores únicos
# junto con la frecuencia de cada uno (value_counts) para detectar
# variantes como 'Yes'/'yes'/' Yes ' que deberían ser la misma categoría
for col in columnas_categoricas:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))  # dropna=False para también ver si hay NaN


Columnas categóricas a revisar: ['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn']
------------------------------------------------------------

--- customerID ---
customerID
3186-AJIEK    1
7590-VHVEG    1
5575-GNVDE    1
8775-CEBBJ    1
2823-LKABH    1
             ..
6713-OKOMC    1
1452-KIOVK    1
9305-CDSKC    1
9237-HQITU    1
7795-CFOCW    1
Name: count, Length: 7043, dtype: int64

--- gender ---
gender
Male      3555
Female    3488
Name: count, dtype: int64

--- Partner ---
Partner
No     3641
Yes    3402
Name: count, dtype: int64

--- Dependents ---
Dependents
No     4933
Yes    2110
Name: count, dtype: int64

--- PhoneService ---
PhoneService
Yes    6361
No      682
Name: count, dtype: int64

--- MultipleLines ---
MultipleLines
No                  3390
Yes            

**1.2 — Conclusión (responde aquí en Markdown):**

¿Encontraste alguna columna con inconsistencias de formato? Si sí, ¿cuál y qué código usarías para corregirla? Si no encontraste ninguna, dilo explícitamente — es una conclusión válida siempre que esté respaldada por lo que revisaste en 1.1.

_Tu respuesta:_

Revisé las 17 columnas categóricas (excluyendo `customerID`, que es un identificador único, y `TotalCharges`, que es numérica mal tipada) con `.value_counts()` y **no encontré inconsistencias de formato**. Todas las columnas binarias (`Partner`, `Dependents`, `PhoneService`, `PaperlessBilling`, `Churn`, `gender`) tienen exactamente 2 categorías bien escritas (`'Yes'`/`'No'` o `'Male'`/`'Female'`), sin variantes de mayúsculas/minúsculas (`'yes'`, `'YES'`), sin espacios extra al inicio o final, y sin errores tipográficos. Las columnas con 3-4 categorías (`Contract`, `PaymentMethod`, `InternetService`, `MultipleLines`, `OnlineSecurity`, etc.) tampoco muestran variantes: cada categoría aparece con una sola grafía consistente en todo el dataset.

Esto tiene sentido porque este dataset parece haber sido generado o limpiado de forma controlada por la fuente (IBM/Telco), sin captura manual de texto libre, que es donde normalmente aparecen errores tipográficos.

Si en el futuro apareciera una inconsistencia como `' Yes'` vs `'Yes'`, el código para corregirla sería:
```python
df['columna'] = df['columna'].str.strip().str.capitalize()
```
pero en este caso, dado lo observado en 1.1, no es necesario aplicar ninguna corrección.


---
## Actividad 2 — Valores inválidos (20 pts)

Varias columnas de este dataset (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`) tienen un tercer valor además de `'Yes'`/`'No'`: `'No internet service'`. De forma similar, `MultipleLines` tiene `'No phone service'`.

In [12]:
# Observamos que OnlineSecurity tiene 3 valores en vez de 2: 'Yes', 'No' y 'No internet service'
df['OnlineSecurity'].value_counts()


,count
OnlineSecurity,
No,3498
Yes,2019
No internet service,1526


In [13]:
# 2.1 — Verificamos si 'No internet service' en OnlineSecurity coincide exactamente
# con las filas donde InternetService == 'No' (es decir, clientes que no tienen
# servicio de internet contratado y por lo tanto no pueden tener OnlineSecurity)

# pd.crosstab cuenta cuántas filas caen en cada combinación de las dos columnas
tabla_cruzada = pd.crosstab(df['OnlineSecurity'], df['InternetService'])
print(tabla_cruzada)

# Verificación explícita: contamos cuántas filas tienen 'No internet service'
# en OnlineSecurity pero SÍ tienen un InternetService distinto de 'No'
# (si el resultado es 0, confirma que la relación es perfecta y consistente)
inconsistencias = df[(df['OnlineSecurity'] == 'No internet service') &
                      (df['InternetService'] != 'No')]
print(f"\nFilas inconsistentes (deberían ser 0): {len(inconsistencias)}")

# Repetimos la misma verificación para MultipleLines vs PhoneService,
# ya que el enunciado menciona el mismo patrón con 'No phone service'
tabla_cruzada_2 = pd.crosstab(df['MultipleLines'], df['PhoneService'])
print("\n", tabla_cruzada_2)

inconsistencias_2 = df[(df['MultipleLines'] == 'No phone service') &
                        (df['PhoneService'] != 'No')]
print(f"\nFilas inconsistentes en MultipleLines (deberían ser 0): {len(inconsistencias_2)}")


InternetService       DSL  Fiber optic    No
OnlineSecurity                              
No                   1241         2257     0
No internet service     0            0  1526
Yes                  1180          839     0

Filas inconsistentes (deberían ser 0): 0

 PhoneService       No   Yes
MultipleLines              
No                  0  3390
No phone service  682     0
Yes                 0  2971

Filas inconsistentes en MultipleLines (deberían ser 0): 0


**2.2 — Conclusión (responde aquí en Markdown):**

¿`'No internet service'` es un valor inválido (como `Absurd`/`YOLO` en la sesión de clase) o es una categoría legítima? Justifica tu respuesta con lo que verificaste en 2.1. ¿Tomarías alguna acción sobre esta columna, o la dejarías tal cual?

_Tu respuesta:_

`'No internet service'` **no es un valor inválido**, es una categoría legítima y consistente con el resto de los datos. La tabla cruzada en 2.1 muestra que las 1,526 filas con `OnlineSecurity == 'No internet service'` corresponden exactamente (0 inconsistencias) a las filas donde `InternetService == 'No'`. Es decir, este valor no es ruido ni un error de captura como `Absurd` o `YOLO` en el ejemplo de clase, sino información derivada de forma lógica: si un cliente no contrató servicio de internet, es imposible que tenga (o no tenga) seguridad en línea — la pregunta ni siquiera aplica para ese cliente. Lo mismo ocurre con `'No phone service'` en `MultipleLines`, que coincide perfectamente con `PhoneService == 'No'`.

Dicho esto, sí consideraría una acción, pero no de "limpieza" sino de **codificación pensando en el modelado**: si estas columnas se van a usar como predictoras en un modelo, dejar 3 categorías (`Yes`/`No`/`No internet service`) puede ser razonable porque preserva la información de "no aplica"; sin embargo, en algunos casos conviene separarla en dos señales:

(a) si tiene o no internet (ya capturado por `InternetService`) y

(b) si tiene el servicio adicional, tratando `No internet service` como equivalente a `No` para esa sub-pregunta, evitando así introducir una categoría redundante con `InternetService`.

Por ahora, para efectos de esta evaluación, **dejaría la columna tal cual**, ya que el valor está justificado y no representa un error.


---
## Actividad 3 — Alta cardinalidad (25 pts)

In [14]:
# 3.1 — Calculamos .nunique() para TODAS las columnas del dataset
# (no solo las categóricas) y la razón (valores únicos / total de filas)

n_filas = len(df)  # total de filas del dataset, usado como denominador de la razón

# Construimos un DataFrame resumen con: nombre de columna, valores únicos y razón de cardinalidad
resumen_cardinalidad = pd.DataFrame({
    'columna': df.columns,
    'valores_unicos': [df[col].nunique() for col in df.columns],
})
resumen_cardinalidad['razon_cardinalidad'] = resumen_cardinalidad['valores_unicos'] / n_filas

# Ordenamos de mayor a menor razón de cardinalidad para identificar rápidamente
# las columnas más problemáticas (más cercanas a 1.0 = un valor único por fila)
resumen_cardinalidad = resumen_cardinalidad.sort_values('razon_cardinalidad', ascending=False)
print(resumen_cardinalidad.to_string(index=False))


         columna  valores_unicos  razon_cardinalidad
      customerID            7043            1.000000
    TotalCharges            6531            0.927304
  MonthlyCharges            1585            0.225046
          tenure              73            0.010365
   PaymentMethod               4            0.000568
 StreamingMovies               3            0.000426
     TechSupport               3            0.000426
    OnlineBackup               3            0.000426
     StreamingTV               3            0.000426
DeviceProtection               3            0.000426
   MultipleLines               3            0.000426
 InternetService               3            0.000426
  OnlineSecurity               3            0.000426
        Contract               3            0.000426
         Partner               2            0.000284
   SeniorCitizen               2            0.000284
          gender               2            0.000284
      Dependents               2            0.

**3.2 — Conclusión (responde aquí en Markdown):**

¿Qué columna(s) tienen alta cardinalidad? Para la columna con mayor cardinalidad: ¿por qué nunca deberías usarla como variable predictora en un modelo, incluso si la codificaras? (relaciona tu respuesta con lo discutido en clase sobre identificadores únicos)

_Tu respuesta:_

Según la tabla de 3.1, la columna con **mayor cardinalidad por mucho** es `customerID`, con 7,043 valores únicos sobre 7,043 filas totales — una razón de cardinalidad de **1.0 (100%)**. Ninguna otra columna se le acerca: la siguiente es `TotalCharges` con una razón de ~0.93 (pero esa es numérica continua mal tipada como texto, no categórica real), y luego `MonthlyCharges` con ~0.23. El resto de las columnas categóricas (`Contract`, `PaymentMethod`, los servicios "Yes/No/No internet service", etc.) tienen cardinalidad muy baja (2 a 4 categorías), por lo que no representan un problema.

`customerID` nunca debería usarse como variable predictora, incluso si se codificara (por ejemplo con one-hot o label encoding), porque **es un identificador único**: cada cliente tiene exactamente un valor y ese valor no se repite ni tiene significado semántico compartido entre clientes. Un modelo que "aprendiera" a partir de `customerID` estaría memorizando el identificador de cada fila del set de entrenamiento en lugar de aprender un patrón generalizable — esto es la definición de **overfitting por identificador único**: no existe ninguna relación causal o correlacional real entre el string `'7590-VHVEG'` y la probabilidad de que un cliente cancele el servicio (`Churn`); cualquier "señal" que el modelo encontrara ahí sería ruido o una fuga de información (leakage) del propio proceso de asignación de IDs, y el modelo no podría generalizar a clientes nuevos con IDs distintos.


**3.3 — Agrupación "Top 10 + Otros"**

En clase agrupaste `country` de Netflix Titles en sus 10 categorías más frecuentes + `'Otros'`, reduciendo su cardinalidad. Aplica la misma técnica aquí sobre la columna de mayor cardinalidad que identificaste en 3.1 (pista: `.value_counts().head(10)`, luego `.where()` + `.isin()`, igual que en el notebook de clase).

In [15]:
# Aplicamos el agrupamiento "Top 10 + Otros" sobre la columna de mayor cardinalidad
# identificada en 3.1: customerID (razón de cardinalidad = 1.0)

# Igual que con 'country' en el notebook de clase, obtenemos las 10 categorías
# (en este caso, los 10 IDs) más frecuentes con value_counts().head(10)
top10_ids = df['customerID'].value_counts().head(10).index
print("Top 10 valores más frecuentes de customerID:")
print(df['customerID'].value_counts().head(10))

# .where(condición, otro_valor): conserva el valor original donde la condición es
# True (el ID está en el top 10) y lo reemplaza por 'Otros' donde es False
df['customerID_agrupado'] = df['customerID'].where(
    df['customerID'].isin(top10_ids), 'Otros'
)

# Revisamos el resultado del agrupamiento
print("\nDistribución después de agrupar:")
print(df['customerID_agrupado'].value_counts().head(12))

# Calculamos qué porcentaje de las filas terminó en la categoría 'Otros'
porcentaje_otros = (df['customerID_agrupado'] == 'Otros').mean() * 100
print(f"\nPorcentaje de filas agrupadas como 'Otros': {porcentaje_otros:.2f}%")


Top 10 valores más frecuentes de customerID:
customerID
3186-AJIEK    1
7590-VHVEG    1
5575-GNVDE    1
8775-CEBBJ    1
2823-LKABH    1
7398-LXGYX    1
1035-IPQPU    1
7203-OYKCT    1
1699-HPSBG    1
9837-FWLCH    1
Name: count, dtype: int64

Distribución después de agrupar:
customerID_agrupado
Otros         7033
7590-VHVEG       1
5575-GNVDE       1
9837-FWLCH       1
1699-HPSBG       1
7203-OYKCT       1
1035-IPQPU       1
7398-LXGYX       1
2823-LKABH       1
8775-CEBBJ       1
3186-AJIEK       1
Name: count, dtype: int64

Porcentaje de filas agrupadas como 'Otros': 99.86%


**3.4 — Conclusión (responde aquí en Markdown):**

Después de agrupar, ¿la columna resultante te parece útil para un modelo? Compara este caso con el de `country` en el notebook de clase: ¿por qué agrupar en "Top 10 + Otros" funciona bien para una variable como `country`, pero no resuelve el problema real de la columna que agrupaste aquí?

_Tu respuesta:_

La columna resultante `customerID_agrupado` **no es útil en absoluto** para un modelo: el resultado del código en 3.3 muestra que el 99.86% de las filas (7,033 de 7,043) cayeron en la categoría `'Otros'`, y las 10 categorías restantes tienen exactamente 1 registro cada una. En la práctica, esta transformación colapsó la columna en una variable con varianza casi nula (todo es `'Otros'` salvo 10 filas), lo cual la vuelve inútil como predictora: un modelo no puede aprender nada de una columna donde el 99.86% de las observaciones comparten el mismo valor.

La razón de esta diferencia con `country` en el notebook de clase es la **naturaleza de la variable, no solo su cardinalidad numérica**: `country` es una variable con **categorías que se repiten de forma significativa** — muchos títulos de Netflix comparten el mismo país de origen, así que aunque existan decenas de países distintos, unos pocos (Estados Unidos, India, Reino Unido, etc.) concentran la mayoría de las observaciones, y agrupar el resto como `'Otros'` preserva la señal real (el patrón de qué países dominan el catálogo) sin perder mucha información. En cambio, `customerID` es, por definición, un **identificador único sin repeticiones** — cada valor aparece exactamente una vez porque esa es su función (identificar de forma unívoca a cada cliente). Agrupar "Top 10 + Otros" reduce el número de categorías, pero **no resuelve el problema de fondo**: el problema no era la cantidad de categorías, sino que la columna nunca tuvo información generalizable que agrupar. Es como recortar las hojas de un árbol sin raíz: la técnica de reducción de cardinalidad soluciona columnas con alta cardinalidad *pero con estructura repetida* (como `country`), no columnas que son identificadores únicos por diseño.


---
## Actividad 4 — Tipos de dato (30 pts)

In [16]:
# TotalCharges debería ser numérica (representa montos en dólares), pero pandas
# la infirió como texto (object/str). Investigamos por qué en 4.1.
df['TotalCharges'].dtype


dtype('O')

**4.1 — Investiga (responde en Markdown):**

`TotalCharges` contiene valores numéricos (montos en dólares), pero pandas la cargó como `object`, no como `float`. Investiga por qué — revisa si hay algún valor que no se vea como un número normal.

_Tu respuesta:_

Al intentar convertir la columna con `pd.to_numeric(df['TotalCharges'], errors='coerce')` y filtrar las filas que se vuelven `NaN`, encontré la causa raíz: **11 filas tienen una cadena vacía (espacio en blanco `' '`) en lugar de un número** en `TotalCharges`. Como al menos un valor de la columna no es convertible a número, pandas no puede asignarle un dtype numérico a toda la columna y la deja como texto (`object`), aunque el 99.8% de los valores sí parezcan números normales (por ejemplo `'29.85'`).

Al revisar esas 11 filas encontré que **todas tienen `tenure == 0`** (cero meses de antigüedad como cliente). Esto tiene una explicación lógica de negocio: son clientes que se acaban de dar de alta y todavía no han completado ni un ciclo de facturación, por lo que su cargo total acumulado (`TotalCharges`) todavía no existe — en vez de dejarlo como `NaN` o `0` desde el origen, el proceso que generó estos datos dejó un espacio en blanco. No es un error tipográfico aleatorio, sino un caso de "dato ausente por diseño" (el cliente es demasiado nuevo para tener un total). Esto justifica por qué la corrección correcta en 4.2 no es simplemente eliminar esas filas, sino imputarlas con 0 (ya que, efectivamente, no han pagado nada todavía).


In [17]:
# 4.2 — Corregimos el tipo de dato de TotalCharges

# pd.to_numeric con errors='coerce' convierte a NaN cualquier valor que no pueda
# transformarse a número (en este caso, las 11 cadenas de espacio en blanco
# detectadas en 4.1), en lugar de lanzar un error y detener la ejecución
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Verificamos cuántos NaN quedaron (deberían ser los mismos 11 identificados en 4.1)
print("Valores nulos tras la conversión:", df['TotalCharges'].isna().sum())

# Como concluimos en 4.1 que estos 11 casos corresponden a clientes con tenure == 0
# (aún no han sido facturados), la imputación correcta -que no pierde información ni
# introduce un sesgo artificial- es rellenarlos con 0, ya que su cargo acumulado
# real es efectivamente cero
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Confirmamos que ya no quedan nulos y que el dtype ahora es numérico (float)
print("Valores nulos después de imputar:", df['TotalCharges'].isna().sum())
print("Nuevo dtype de TotalCharges:", df['TotalCharges'].dtype)


Valores nulos tras la conversión: 11
Valores nulos después de imputar: 0
Nuevo dtype de TotalCharges: float64


In [18]:
# 4.3 — Convertimos a category las columnas categóricas de baja cardinalidad
# y valores fijos, identificadas en la Actividad 3 (todas con razón de
# cardinalidad menor a 0.001 y un conjunto cerrado de categorías: 2 a 4 valores)

columnas_a_categoria = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
    'PaperlessBilling', 'PaymentMethod', 'Churn'
]
# Nota: NO incluimos customerID (cardinalidad 1.0, es un identificador, no una
# categoría con valores repetidos) ni TotalCharges/MonthlyCharges (numéricas continuas)

# astype('category') le indica a pandas que use una codificación interna eficiente
# (basada en códigos enteros) en lugar de guardar cada string por separado,
# lo cual ahorra memoria y acelera operaciones de agrupamiento/filtrado
for col in columnas_a_categoria:
    df[col] = df[col].astype('category')

# Verificamos con .dtypes que el cambio se aplicó correctamente
print(df.dtypes)


customerID               object
gender                 category
SeniorCitizen             int64
Partner                category
Dependents             category
tenure                    int64
PhoneService           category
MultipleLines          category
InternetService        category
OnlineSecurity         category
OnlineBackup           category
DeviceProtection       category
TechSupport            category
StreamingTV            category
StreamingMovies        category
Contract               category
PaperlessBilling       category
PaymentMethod          category
MonthlyCharges          float64
TotalCharges            float64
Churn                  category
customerID_agrupado      object
dtype: object


---
## Reflexión final (10 pts)

Con base en las 4 actividades anteriores, responde:

1. De las alertas que detectaste (formato, valores inválidos, cardinalidad, tipos), ¿cuál te pareció más fácil de decidir y cuál más difícil? ¿Por qué?
2. Si tuvieras que entregar este dataset ya "perfilado" a un compañero para que construya un modelo predictivo, ¿qué le dirías sobre `customerID` y sobre `TotalCharges`?

_Tu respuesta:_

**1.** La alerta más fácil de decidir fue la de **formato y valores inconsistentes** (Actividad 1): con solo recorrer `.value_counts()` de cada columna quedó claro, sin ambigüedad, que no había variantes de mayúsculas ni espacios — el dataset ya venía limpio en ese aspecto, y no requería juicio adicional.

La más difícil fue la de **valores inválidos** (Actividad 2), específicamente decidir qué hacer con `'No internet service'`. A diferencia de un valor claramente erróneo como `Absurd` o `YOLO` (que se ve en clase), aquí el valor *parece* inválido a primera vista (es una tercera categoría inesperada en lo que debería ser binario Yes/No), pero al cruzarlo con `InternetService` se revela que es una categoría legítima y consistente. Esto obligó a verificar con evidencia (crosstab) en lugar de decidir solo por intuición, y a razonar sobre el **motivo de negocio** detrás del dato (no aplica vs. error), que es un tipo de juicio más sutil que simplemente detectar una grafía distinta.

**2.** Le diría a mi compañero:
- **`customerID`**: no debe usarse jamás como variable predictora, sin importar cómo se codifique (one-hot, label encoding, hashing, etc.). Es un identificador único (cardinalidad = 1.0) y cualquier "patrón" que un modelo aprenda de él es memorización/overfitting sin capacidad de generalizar a clientes nuevos. Puede conservarse solo como llave para hacer join con otras tablas o para trazabilidad, pero debe excluirse explícitamente de la matriz de features (`X`) antes de entrenar cualquier modelo.
- **`TotalCharges`**: llegó mal tipada como texto porque 11 filas (clientes con `tenure == 0`, recién dados de alta) tenían un espacio en blanco en vez de un número. Ya la convertí a `float` con `pd.to_numeric(errors='coerce')` y esas 11 filas quedaron imputadas en 0, lo cual es consistente con la realidad de negocio (aún no han sido facturados). Le advertiría que revise si esa imputación en 0 tiene sentido para el modelo que va a construir: si el objetivo es predecir `Churn`, un cliente con `tenure=0` y `TotalCharges=0` es coherente y no distorsiona el análisis, pero si usara otra imputación (como la media) introduciría un sesgo, ya que estos clientes genuinamente no han generado cargos todavía.


---
## Rúbrica de evaluación

| Actividad | Puntos | Criterio |
|---|---|---|
| 1. Formato y valores inconsistentes | 15 | Revisó todas las columnas categóricas (no solo una); código comentado y conclusión (1.2) respaldada por lo que se observó, no solo afirmada |
| 2. Valores inválidos | 20 | Verificó la relación entre columnas antes de concluir; la conclusión (2.2) justifica con evidencia, no solo con intuición |
| 3. Alta cardinalidad | 25 | Calcula `.nunique()` para todas las columnas; aplica correctamente el agrupamiento Top 10 + Otros; la conclusión (3.4) explica por qué agrupar no resuelve el problema de un identificador único |
| 4. Tipos de dato | 30 | Identifica la causa raíz del `dtype` incorrecto (4.1); corrige `TotalCharges` sin perder información; conversión a `category` justificada por cardinalidad, no aplicada al azar |
| Reflexión final | 10 | Conecta las 4 actividades entre sí; no es una respuesta genérica o intercambiable con cualquier dataset |
| **Total** | **100** | |

**Nota sobre las conclusiones:** cada actividad tiene su propia pregunta de conclusión (1.2, 2.2, 3.4, 4.1) — esas respuestas se califican como parte de la actividad correspondiente, no solo la Reflexión final. Comenta tu código donde tomes una decisión (por ejemplo, por qué elegiste cierto umbral o cierta corrección) — el comentario también cuenta dentro del puntaje de cada actividad.